# jobs

> One stored row for each background job: what runs, what ended, and where to see it. The activity chip, the event stream and the Mac app read these rows.

abhyasa does slow work in the background: it looks for sources, makes a walk from a link or a goal, files a folder, plans a course,
and writes the notebook of a step. Each such job has one row here. A row has a `kind`, a `state` (`running`, `done`, `failed`,
`stopped` or `closed`), a `said` line, the `start` and `end` times, a `title`, and a `url`: the page of its object.

The rows are on one document of the learning shelf, so they survive a restart. The pool future of a job stays in memory; the row is
the truth for the pages. At a start, each row that still says `running` becomes `closed`: no job lives across a restart.

In [ ]:
#| default_exp jobs

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

The rows live on the learning shelf of the vault, next to the concepts and the walks.

In [ ]:
#| export
import json, threading, time
from fastcore.utils import *
from abhyasa.store import Learning, _src

## The row

`ENDED` are the states of a job that does not run. `FRESH` is how long an ended job shows in the activity chip, until you open its
page. `KEEP` is how long an ended row stays at all.

In [ ]:
#| export
ENDED = ('done', 'failed', 'stopped', 'closed')
FRESH = 6 * 3600   # seconds that an ended job shows in the activity chip, until you open its page
KEEP = 2 * 86400   # seconds that an ended row stays, then it goes
_lock = threading.RLock()   # one change of the rows at a time
_ver = [0]   # goes up at each change, so the event stream knows that a row changed

The rows are the meta of one document. Its text never changes, so a write changes only the meta.

In [ ]:
#| export
def _doc(lr:Learning): return first(lr.v.t.docs(where='source=?', where_args=[_src('jobs', 'all')]))

def _write(lr:Learning, rows:dict):
    m = dict(rows=rows)
    r = lr.v.add('The background jobs of abhyasa.', title='abhyasa jobs', source=_src('jobs', 'all'), kind='jobs', meta=m)
    if not r.get('changed'): lr.v.set_meta(r['doc_id'], **m)
    _ver[0] += 1

@patch
def jobs(self:Learning) -> dict:
    "Each job row by its key, as an `AttrDict` with its `key`."
    r = _doc(self)
    rows = json.loads(r['meta'] or '{}').get('rows') or {} if r else {}
    return {k: AttrDict(v, key=k) for k, v in rows.items()}

@patch
def job(self:Learning, key:str) -> AttrDict|None:
    "The job row `key`, or None."
    return self.jobs().get(key)

A change merges into one row. An ended row older than `KEEP` goes at the same time.

In [ ]:
#| export
@patch
def put_job(self:Learning, key:str, **kv) -> AttrDict:
    "Merge `kv` into the job row `key` and save it. Ended rows older than `KEEP` go."
    with _lock:
        now, rows = time.time(), {k: {x: y for x, y in v.items() if x != 'key'} for k, v in self.jobs().items()}
        rows[key] = {**rows.get(key, {}), **kv}
        _write(self, {k: v for k, v in rows.items() if v.get('state') == 'running' or now - (v.get('end') or now) < KEEP})
        return AttrDict(rows[key], key=key)

def job_version() -> int:
    "A count that goes up at each change of a row in this run."
    return _ver[0]

## Start, end, and a pool job

`job_start` writes a running row. `obj` is the object of the job, such as `walk:<id>` or `course:<id>`; its page marks the end as
seen. `quiet` keeps a job out of the chip, such as the research of a side walk of a talk. `job_end` closes it.

In [ ]:
#| export
def job_start(lr:Learning, key:str, kind:str, title:str='', said:str='', url:str='', obj:str='', quiet:bool=False) -> AttrDict:
    "Note that the job `key` of `kind` runs now, with its `title`, its `said` line, the `url` of its page and its object `obj`."
    return lr.put_job(key, kind=kind, title=title, said=said, url=url, obj=obj, quiet=quiet, state='running', start=time.time(), end=None, why='', seen=False)

def job_end(lr:Learning, key:str, state:str='done', why:str='', **kv) -> AttrDict|None:
    "Note that the job `key` ended now in `state`, with `why` for a failure. `kv` can change its `said`, `url` or `obj`. None when there is no row."
    if lr.job(key) is None: return None
    return lr.put_job(key, state=state, end=time.time(), why=why, **kv)

`track` gives a pool future its row: running now, then `done` or `failed` when it ends. `ok` maps the result to more fields of the row,
such as the link of the walk that the job made.

In [ ]:
#| export
def track(lr:Learning, key:str, fut, kind:str, ok=None, why=None, **kw):
    "Start the row `key` of the pool future `fut`, and end it when `fut` ends: `ok(result)` gives more fields, `why(error)` the line of a failure. Return `fut`."
    job_start(lr, key, kind, **kw)
    def end(f):
        try:
            if (e := f.exception()) is not None: job_end(lr, key, 'failed', why=why(e) if why else 'It did not work.')
            else: job_end(lr, key, 'done', **((ok(f.result()) or {}) if ok else {}))
        except Exception: pass   # a row that cannot be written never breaks the job
    fut.add_done_callback(end)
    return fut

A job that runs long says what it does now. `Said` is the state of one pool job, and a new `said` goes to its row too, so a
page that reads the row sees the step.

In [ ]:
#| export
class Said(AttrDict):
    "The state of one pool job, for its card; `update(said=…)` also writes the line to the row `key` of `lr`."
    def __init__(self, lr:Learning, key:str, **kw): super().__init__(**kw); self.__dict__.update(_lr=lr, _key=key)
    def update(self, *a, **kw):
        super().update(*a, **kw)
        if 'said' in kw and (r := self._lr.job(self._key)) is not None and r.state == 'running':
            try: self._lr.put_job(self._key, said=kw['said'])
            except Exception: pass

## Seen, closed, and the activity

Opening the page of an object marks its ended jobs as seen. At a start of the app, a row that says `running` has no job any more:
it becomes `closed`.

In [ ]:
#| export
def seen_jobs(lr:Learning, obj:str) -> int:
    "Mark each ended job of the object `obj` as seen. Return the count that changed."
    with _lock:
        ks = [k for k, r in lr.jobs().items() if r.get('obj') == obj and r.state in ENDED and not r.get('seen')]
        for k in ks: lr.put_job(k, seen=True)
        return len(ks)

def close_jobs(lr:Learning) -> int:
    "At a start: each row that says `running` becomes `closed`, as its job did not live across the restart. Return the count."
    with _lock:
        ks = [k for k, r in lr.jobs().items() if r.state == 'running']
        for k in ks: lr.put_job(k, state='closed', end=time.time())
        return len(ks)

`activity` is what the chip lists: each job that runs, then each ended job that is `done`, `failed` or `closed`, not seen, and
less than `FRESH` old. A job that you stopped is not news. A quiet job never shows.

In [ ]:
#| export
def activity(lr:Learning, now:float=None) -> L:
    "The rows for the activity chip: the running jobs, newest first, then the ended jobs that are new to you."
    now, rs = now or time.time(), [r for r in lr.jobs().values() if not r.get('quiet')]
    run = sorted((r for r in rs if r.state == 'running'), key=lambda r: -(r.get('start') or 0))
    new = sorted((r for r in rs if r.state in ('done', 'failed', 'closed') and not r.get('seen') and now - (r.get('end') or 0) < FRESH), key=lambda r: -(r.get('end') or 0))
    return L(run + new)

WORD = dict(done='Ready', failed='Failed', closed='Stopped')

def chip_words(rows) -> str:
    "The words of the chip: `2 running` while a job runs, else `Ready: {title}`, `Failed: {title}` or `Stopped: {title}`, with the count of the others."
    if not rows: return ''
    if (n := sum(r.state == 'running' for r in rows)): return f'{n} running'
    r = rows[0]
    return f"{WORD.get(r.state, 'Ready')}: {r.get('title') or 'a job'}" + (f' and {len(rows) - 1} more' if len(rows) > 1 else '')

## Tests

A temporary vault for the tests.

In [ ]:
from abhyasa.store import temp_learning

A job starts with a running row and no end; the row is in the chip.

In [ ]:
lr = temp_learning()
r = job_start(lr, 'walk:w1:research', 'research', title='Softmax', said='Finding sources', url='/w/w1#sources', obj='walk:w1')
test_eq((r.state, r.end, r.key), ('running', None, 'walk:w1:research'))
test_eq([x.key for x in activity(lr)], ['walk:w1:research'])
test_eq(chip_words(activity(lr)), '1 running')

Its end is news until you open the walk.

In [ ]:
v = job_version()
job_end(lr, 'walk:w1:research', said='2 of 3 ideas have a source')
assert job_version() > v
test_eq(chip_words(activity(lr)), 'Ready: Softmax')
test_eq(seen_jobs(lr, 'walk:w1'), 1)
test_eq(activity(lr), [])

An old end and a quiet job do not show; a job that you stopped is not news.

In [ ]:
job_start(lr, 'k2', 'build', title='Old'); job_end(lr, 'k2')
test_eq(activity(lr, now=time.time() + FRESH + 1), [])
job_start(lr, 'k3', 'research', title='Talk', quiet=True)
job_start(lr, 'k4', 'research', title='Mine'); job_end(lr, 'k4', 'stopped')
test_eq([x.key for x in activity(lr)], ['k2'])   # k2 is new now; k3 is quiet and k4 was stopped
test_eq(job_end(lr, 'nope'), None)

A restart closes each running row, and the chip says so.

In [ ]:
job_start(lr, 'k5', 'build', title='Diffusion')
test_eq(close_jobs(lr), 2)   # k3 and k5
test_eq(lr.job('k5').state, 'closed')
test_eq(chip_words(activity(lr)), 'Stopped: Diffusion and 1 more')

`track` ends the row of a pool future: `done` with the fields of `ok`, or `failed` with the line of `why`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

In [ ]:
pool = ThreadPoolExecutor(1)
track(lr, 'b1', pool.submit(lambda: 'w9'), 'build', ok=lambda jid: dict(url=f'/w/{jid}', obj=f'walk:{jid}'), title='A link').result()
pool.shutdown(wait=True)
test_eq((lr.job('b1').state, lr.job('b1').url, lr.job('b1').obj), ('done', '/w/w9', 'walk:w9'))
pool = ThreadPoolExecutor(1)
f = track(lr, 'b2', pool.submit(lambda: 1 / 0), 'build', why=lambda e: 'No walk.', title='Bad')
pool.shutdown(wait=True)
test_eq((lr.job('b2').state, lr.job('b2').why), ('failed', 'No walk.'))

A new line of a running job goes to its row.

In [ ]:
job_start(lr, 'b3', 'build', title='Paper', said='Reading the paper…')
st = Said(lr, 'b3', said='Reading the paper…')
st.update(said='Reading 2 of 3 pages…')
test_eq((st.said, lr.job('b3').said), ('Reading 2 of 3 pages…', 'Reading 2 of 3 pages…'))

An ended row older than `KEEP` goes at the next change.

In [ ]:
lr.put_job('b2', end=time.time() - KEEP - 1)
lr.put_job('b3', said='Reading 3 of 3 pages…')
assert lr.job('b2') is None and lr.job('b3') is not None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()